# App 5 · MCP — 让一份 server 在所有 LLM 之间通用

2024 年中之前，要让一个内部 CRM 的 `query_order` 函数被 GPT-4 调到，你得给 OpenAI 写一份 `function_call` schema；要让 Claude 也能调，再写一份 `tool_use` schema；接 Cursor IDE 又得写一份 Cursor 自己的格式。同一个函数，三套描述，每次新模型出来都得重写。

Anthropic 在 2024 年 11 月放出 [MCP（Model Context Protocol）](https://modelcontextprotocol.io/) 就是为了终结这件事。它没发明新算法、新模型，只是定义了一套**工具调用的传输协议**：写一次 server，所有支持 MCP 的 LLM 客户端（Claude Desktop、Cursor、Claude Code、自家 app）都能跟它对话。

这一节我们做三件事——把 MCP 的三类原语（Tools / Resources / Prompts）讲清楚；用纯 Python 跑一遍真协议（subprocess + stdio JSON-RPC，看到完整的 initialize / tools/list / tools/call / shutdown 帧）；最后加上权限和容错。这两件事是把 demo 推到生产前必补的功课。

> **跑这一节前**：跑过 [App0](./App0_Setup_Check.ipynb) 把环境就绪，再跑过 App1–App4 理解 Agent 的工具调用语义。可选 SDK `pip install 'mcp>=0.9'` 不装也行——`utils/mcp_helpers.py` 里有一个 in-process 教学版 `EduMCPServer` 可以兜底，API 跟官方 SDK 故意保持一致，后面切换只是 `pip install` 一行的事。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"[DIR] repo root: {_root}")


[DIR] repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code


In [2]:
# 导入：LLM + MCP + Skills helpers
from utils.config import setup
env = setup()
from utils.mcp_helpers import (
    EduMCPServer, EduMCPClient,
    ToolDef, ResourceDef, PromptDef,
    tool_from_function, MCP_AVAILABLE,
)
from utils.skills_helpers import (
    Skill, parse_skill_md, validate_skill,
    discover_skills, match_skill_for_query, load_skill_progressive,
)
import json

llm = env.get_llm()
print(f"OK LLM 就位")
print(f"OK MCP SDK 可用: {MCP_AVAILABLE}  (False 走 EduMCPServer 教学模式)")
print(f"OK Skills helpers 就位")


[OK] 已加载配置: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\.env
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus-2025-01-25
  Embedding: dashscope / text-embedding-v3


[LLM] dashscope / qwen-plus-2025-01-25
OK LLM 就位
OK MCP SDK 可用: True  (False 走 EduMCPServer 教学模式)
OK Skills helpers 就位


---


## Why MCP

### MCP 出现前的集成问题

每家 LLM 都有自己的 tool 调用方式：

| 厂商 | 调用方式 |
|---|---|
| OpenAI | `function_call` |
| Anthropic | `tool_use` |
| Google | `function_declarations` |
| Cohere | `connectors` |

**结果**：你要让 LLM 调你公司的 CRM API，要为 N 家 LLM 各写一遍 schema。

### MCP = Model Context Protocol

Anthropic 2024 末推出的**开放式跨厂工具调用协议**；到 2025-2026，很多 Agent/IDE 生态开始围绕它做工具接入。

可以把它理解为统一工具接入层：写一次 server，所有支持 MCP 的 LLM/IDE 都能用。

### 三件套

```
MCP Server 暴露：
├── Tools      — LLM 可调用的函数（带 JSON schema）
├── Resources  — LLM 可读取的数据源（file / db / api）
└── Prompts    — 可复用的提示模板（带参数）
```

**今天聚焦 Tools + Resources（Prompts 一句话提）。**


---

## Tools

Tool = 一个函数 + description + JSON schema。LLM 看 description 决定调不调；调时按 schema 填参数。


In [3]:
# 用 helper 把普通 Python 函数自动转成 ToolDef
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

add_tool = tool_from_function(add)

print("ToolDef 自动生成：")
print(f"  name: {add_tool.name}")
print(f"  description: {add_tool.description}")
print(f"  parameters schema: {json.dumps(add_tool.parameters, indent=2)}")
print(f"\n  call(a=3, b=5) → {add_tool.call(a=3, b=5)}")


ToolDef 自动生成：
  name: add
  description: Add two integers.
  parameters schema: {
  "type": "object",
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ]
}

  call(a=3, b=5) → 8


In [4]:
# 把多个 Tool 放进 Server，让 LLM 当 client
def multiply(a: int, b: int, label: str = "result") -> str:
    """Multiply two integers and return labeled result."""
    return f"{label}: {a * b}"

server = EduMCPServer(name="math-helper")
server.add_tool(add_tool)
server.add_tool(tool_from_function(multiply))

# Demo: LLM 看 tools 列表 → 决定调哪个
# Qwen / DashScope 有时会把 arguments 写成 list；这里按 schema 归一化，避免课堂 demo 因格式小偏差失败。
def _extract_json_object(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    start, end = raw.find("{"), raw.rfind("}")
    return raw[start:end + 1] if start >= 0 and end >= start else raw


def _normalize_tool_arguments(tool_schema, arguments):
    props = tool_schema["parameters"].get("properties", {})
    names = list(props.keys())
    if isinstance(arguments, list):
        arguments = {name: value for name, value in zip(names, arguments)}
    elif not isinstance(arguments, dict):
        arguments = {}
    for name, meta in props.items():
        if name not in arguments:
            continue
        value = arguments[name]
        if meta.get("type") == "integer" and isinstance(value, str) and value.strip().lstrip("-").isdigit():
            arguments[name] = int(value)
        elif meta.get("type") == "number" and isinstance(value, str):
            try:
                arguments[name] = float(value)
            except ValueError:
                pass
    return arguments


def llm_calls_tool(query, srv):
    tools = srv.list_tools()
    desc = "\n".join(
        f"- {t['name']} 参数={list(t['parameters']['properties'].keys())}: {t['description']}"
        for t in tools
    )
    raw = llm.generate(
        f'''你是一个 tool router。可用 tools:\n{desc}\n\n用户: {query}\n\n只输出 JSON 对象，格式必须是：\n{{"tool": "工具名", "arguments": {{"参数名": 参数值}}}}\n注意 arguments 必须是 object，不要输出数组。''',
        temperature=0,
    ).strip()
    try:
        plan = json.loads(_extract_json_object(raw))
        tool_schema = next(t for t in tools if t["name"] == plan["tool"])
        arguments = _normalize_tool_arguments(tool_schema, plan.get("arguments", {}))
        return f"{plan['tool']}({arguments}) → {srv.call_tool(plan['tool'], arguments)}"
    except (json.JSONDecodeError, KeyError, ValueError, TypeError, StopIteration) as e:
        return f"[LLM JSON 仍需修正] {e}; raw={raw[:160]}"

print(llm_calls_tool("帮我算 15 加 27", server))
print(llm_calls_tool("把 8 和 9 相乘标记为 'order_total'", server))


add({'a': 15, 'b': 27}) → 42


multiply({'a': 8, 'b': 9, 'label': 'order_total'}) → order_total: 72


In [5]:
# [LEARNER_FILL] 难度=基础+进阶 | 提示=约35-50行 | 参考实现见 enterprise_5days/student/Day4_下午_MCP与Skills.ipynb
# ============================================================
# 练习 1 | 自定义 Tool + Schema 严格校验
# ============================================================
#
# 【基础】（人人必做，10 min）
#   实现 build_search_tool()：定义 search_employee(name, department=None) 函数
#   并包装成 ToolDef 返回
#
# 【进阶】（技术学员选做，10 min）
#   实现 build_strict_tool(func)：在 tool_from_function 基础上加严格校验：
#   - 缺 required 参数 → ValueError("Missing: ...")
#   - 传未声明参数 → ValueError("Unknown: ...")
# ============================================================

EMPLOYEES = [
    {"name": "张三", "department": "技术部", "level": "senior"},
    {"name": "李四", "department": "市场部", "level": "junior"},
    {"name": "王五", "department": "技术部", "level": "lead"},
]


def build_search_tool():
    """【基础】返回 ToolDef"""
    # ↓↓↓ 【基础】填空（约 6 行）↓↓↓
    def search_employee(name: str, department: str = None):
        results = [e for e in EMPLOYEES if name in e["name"]]
        if department:
            results = [e for e in results if e["department"] == department]
        return json.dumps(results, ensure_ascii=False)
    return tool_from_function(search_employee)
    # ↑↑↑ 【基础】结束 ↑↑↑


def build_strict_tool(func):
    """【进阶】带 unknown 参数校验"""
    # ↓↓↓ 【进阶】填空（约 12 行）↓↓↓
    base = tool_from_function(func)
    declared = set(base.parameters["properties"].keys())
    required = set(base.parameters.get("required", []))
    original = base.func
    def strict(**kwargs):
        missing = required - set(kwargs.keys())
        if missing:
            raise ValueError(f"Missing: {sorted(missing)}")
        unknown = set(kwargs.keys()) - declared
        if unknown:
            raise ValueError(f"Unknown: {sorted(unknown)}")
        return original(**kwargs)
    base.func = strict
    return base
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】build_search_tool"); print("=" * 56)
    try:
        tool = build_search_tool()
        assert tool.name == "search_employee"
        result = tool.call(name="张三")
        print(f"  search('张三') → {result}")
        assert "张三" in result
        result = tool.call(name="王", department="技术部")
        print(f"  search('王', dept='技术部') → {result}")
        print("OK 基础通过\n")
    except NotImplementedError:
        print("SKIP 基础未实现\n"); return
    except Exception as e:
        print(f"FAIL 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】build_strict_tool"); print("=" * 56)
    try:
        def divide(a: int, b: int) -> float:
            """Divide a/b"""
            return a / b
        strict = build_strict_tool(divide)
        assert strict.call(a=10, b=2) == 5.0
        print(f"  strict(a=10,b=2) → 5.0 OK")
        try:
            strict.call(a=10)
        except ValueError as e:
            print(f"  缺参报错: {e} OK")
        try:
            strict.call(a=10, b=2, c=99)
        except ValueError as e:
            print(f"  未知参数报错: {e} OK")
        print("OK 进阶通过")
    except NotImplementedError:
        print("SKIP 进阶跳过")
    except Exception as e:
        print(f"FAIL 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】build_search_tool
  search('张三') → [{"name": "张三", "department": "技术部", "level": "senior"}]
  search('王', dept='技术部') → [{"name": "王五", "department": "技术部", "level": "lead"}]
OK 基础通过

【进阶】build_strict_tool
  strict(a=10,b=2) → 5.0 OK
  缺参报错: Missing: ['b'] OK
  未知参数报错: Unknown: ['c'] OK
OK 进阶通过


---

## Resources

Resource = LLM 可**读取的数据源**（不是函数调用）。

| Tool | Resource |
|---|---|
| 函数式（有副作用） | 数据式（只读） |
| 例：`send_email()` | 例：`file:///docs/policy.md` |


In [6]:
# 演示：定义 file resource + dynamic resource
DOCS = {
    "policy_leave.md": "# 请假制度\n年假：5 年以下 5 天/年；5 年以上 15 天/年。",
    "policy_expense.md": "# 报销制度\n餐费 ≤ 100 元/餐。差旅一线 500/晚，二三线 350/晚。",
}

server2 = EduMCPServer(name="enterprise-docs")
for fn in DOCS:
    server2.add_resource(ResourceDef(
        uri=f"file:///docs/{fn}",
        name=fn,
        mime_type="text/markdown",
        reader=(lambda f=fn: DOCS[f]),
    ))

# 加一个动态 resource（每次读返回当前时间戳）
import time, random
server2.add_resource(ResourceDef(
    uri="live:///stats",
    name="live_stats",
    mime_type="application/json",
    reader=lambda: json.dumps({"timestamp": time.time(), "active_users": random.randint(50, 200)}),
))

print("可读取的 Resources:")
for r in server2.list_resources():
    print(f"  • {r['uri']}  ({r['mime_type']})")

print(f"\n读 policy_leave: {server2.read_resource('file:///docs/policy_leave.md')[:60]}...")
print(f"读 live_stats: {server2.read_resource('live:///stats')}")
print("\n提示：Live Resource 让 LLM 看到的总是『现在』，缓存失效问题自动解决")


可读取的 Resources:
  • file:///docs/policy_leave.md  (text/markdown)
  • file:///docs/policy_expense.md  (text/markdown)
  • live:///stats  (application/json)

读 policy_leave: # 请假制度
年假：5 年以下 5 天/年；5 年以上 15 天/年。...
读 live_stats: {"timestamp": 1778041019.7276359, "active_users": 61}

提示：Live Resource 让 LLM 看到的总是『现在』，缓存失效问题自动解决


---

## 实战：写真实 MCP Server + 权限层

我们在 `mcp_server_demo/` 写好了一个**真正可跑的 MCP server**：
- `server.py` — 暴露 3 个企业 tool（订单/库存/通知）
- `client_test.py` — client 测试

下面用 LLM 当大脑试一遍**端到端**流程，并加权限层。


In [7]:
# 复用 mcp_server_demo
import sys
from pathlib import Path
sys.path.insert(0, str(Path('Applications/mcp_server_demo')))
from server import build_server as build_demo_server  # type: ignore

demo_server = build_demo_server()
demo_client = EduMCPClient(user_id="alice")
demo_client.connect(demo_server)

print(f"OK Demo server 就位 ({len(demo_client.list_all_tools())} tools)")
for t in demo_client.list_all_tools():
    print(f"  • {t['name']}: {t['description']}")


OK Demo server 就位 (3 tools)
  • query_order: Look up an order by ID
  • check_inventory: Check stock quantity for a SKU
  • send_notification: Send a notification to a user


In [8]:
# Demo: LLM 用 demo server 完成端到端任务
def llm_use_mcp(query, client):
    tools = client.list_all_tools()
    desc = "\n".join(f"- [{t['server']}] {t['name']}({list(t['parameters']['properties'].keys())}): {t['description']}" for t in tools)
    raw = llm.generate(
        f"可用工具:\n{desc}\n\n用户: {query}\n\n输出 JSON: {{\"server\": \"...\", \"tool\": \"...\", \"arguments\": {{...}}}}。只输出 JSON。",
        temperature=0,
    ).strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        plan = json.loads(raw)
        result = client.call(plan["server"], plan["tool"], **plan["arguments"])
        answer = llm.generate(
            f"用户问 '{query}'，调 {plan['tool']}({plan['arguments']}) 得到: {result}。请用一句话给最终答复。",
            temperature=0.2,
        )
        return {"plan": plan, "raw": result, "answer": answer.strip()}
    except Exception as e:
        return {"error": str(e), "raw_llm": raw[:200]}


print("=" * 60); print("Demo: 查订单"); print("=" * 60)
print(json.dumps(llm_use_mcp("查 ORD-001", demo_client), ensure_ascii=False, indent=2))

print("\n" + "=" * 60); print("Demo: 查库存"); print("=" * 60)
print(json.dumps(llm_use_mcp("SKU-A100 还有多少货", demo_client), ensure_ascii=False, indent=2))


Demo: 查订单


{
  "plan": {
    "server": "enterprise-demo",
    "tool": "query_order",
    "arguments": {
      "order_id": "ORD-001"
    }
  },
  "raw": "{\"status\": \"shipped\", \"total\": 199.0, \"customer\": \"alice\"}",
  "answer": "订单 ORD-001 已发货，总金额为 199.0 元，收件人为 alice。"
}

Demo: 查库存


{
  "plan": {
    "server": "enterprise-demo",
    "tool": "check_inventory",
    "arguments": {
      "sku": "SKU-A100"
    }
  },
  "raw": "{\"sku\": \"SKU-A100\", \"quantity\": 35, \"in_stock\": true}",
  "answer": "SKU-A100 目前有 35 件库存，仍在售。"
}


---

### 真起独立 server 进程：subprocess + stdio JSON-RPC

上面 `llm_use_mcp` 的 demo 把 server 放在**同一个 Python 进程**里——方便教学，但**不是真实生产方式**。

**真 MCP 协议的本质**：
1. server 是**独立进程**（用任何语言写都行，不止 Python）
2. client 用 **subprocess** 拉起 server
3. 双方走 **JSON-RPC over stdio**（每行一个 JSON 消息）
4. 协议方法：`initialize` / `tools/list` / `tools/call` / `resources/list` / `resources/read` / `shutdown`

下面 demo **真起一个独立 server 进程**（`mcp_server_demo/server.py --stdio`），用真 JSON-RPC 与它通信。

> 提示: 这跟 Anthropic 官方 `mcp` Python SDK（需 Python 3.10+）采用的是同一类通信思路。这里手写协议帧，是为了让你看到 SDK 封装下的协议交互。Claude Desktop / Cursor / Claude Code 也是这样跟 MCP server 说话。


In [9]:
# 真起独立 server 进程 + 跑 stdio JSON-RPC 通信
import subprocess, json, time, sys as _sys
from pathlib import Path

server_script = Path("Applications/mcp_server_demo/server.py")

# 1. 起 server subprocess
print(f"启动 server: {_sys.executable} {server_script} --stdio")
proc = subprocess.Popen(
    [_sys.executable, str(server_script), "--stdio"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    encoding="utf-8",
    bufsize=1,
)
time.sleep(0.3)  # 让 server 启动

req_id = 0
def rpc_call(method, params=None):
    global req_id
    req_id += 1
    req = {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params or {}}
    line = json.dumps(req, ensure_ascii=False) + "\n"
    print(f"  → {method}({params or '{}'})")
    proc.stdin.write(line); proc.stdin.flush()
    resp_line = proc.stdout.readline()
    resp = json.loads(resp_line.strip())
    if "error" in resp:
        print(f"  ← ERROR: {resp['error']}")
        return None
    return resp.get("result", {})

# 2. 协议握手
print("\n--- Step 1: initialize 协议握手 ---")
info = rpc_call("initialize")
print(f"  ← server={info['server_info']['name']} v{info['server_info']['version']}, protocol={info['protocol_version']}")

# 3. 列出 tools
print("\n--- Step 2: tools/list 列出能力 ---")
tools = rpc_call("tools/list")
for t in tools["tools"]:
    print(f"  ← {t['name']}: {t['description'][:50]}")

# 4. 真调一个 tool
print("\n--- Step 3: tools/call 调用 query_order ---")
result = rpc_call("tools/call", {"name": "query_order", "arguments": {"order_id": "ORD-002"}})
print(f"  ← {result['content'][0]['text']}")

# 5. 再调另一个
print("\n--- Step 4: tools/call 调用 check_inventory ---")
result = rpc_call("tools/call", {"name": "check_inventory", "arguments": {"sku": "SKU-A100"}})
print(f"  ← {result['content'][0]['text']}")

# 6. 关闭
print("\n--- Step 5: shutdown 关闭 server ---")
rpc_call("shutdown")
proc.terminate()
try:
    proc.wait(timeout=2)
except subprocess.TimeoutExpired:
    proc.kill()

print("\n提示：这就是 Claude Desktop / Cursor / Claude Code 跟 MCP server 通信的真实方式。")
print("   每条 JSON-RPC 消息一行，server 是独立进程（任何语言都能写），双方靠 stdio 管道通信。")


启动 server: E:\conda\envs\llmcs\python.exe Applications\mcp_server_demo\server.py --stdio



--- Step 1: initialize 协议握手 ---
  → initialize({})


  ← server=enterprise-demo v0.1.0, protocol=2025-11-05-edu

--- Step 2: tools/list 列出能力 ---
  → tools/list({})
  ← query_order: Look up an order by ID
  ← check_inventory: Check stock quantity for a SKU
  ← send_notification: Send a notification to a user

--- Step 3: tools/call 调用 query_order ---
  → tools/call({'name': 'query_order', 'arguments': {'order_id': 'ORD-002'}})
  ← {"status": "pending", "total": 89.0, "customer": "bob"}

--- Step 4: tools/call 调用 check_inventory ---
  → tools/call({'name': 'check_inventory', 'arguments': {'sku': 'SKU-A100'}})
  ← {"sku": "SKU-A100", "quantity": 35, "in_stock": true}

--- Step 5: shutdown 关闭 server ---
  → shutdown({})

提示：这就是 Claude Desktop / Cursor / Claude Code 跟 MCP server 通信的真实方式。
   每条 JSON-RPC 消息一行，server 是独立进程（任何语言都能写），双方靠 stdio 管道通信。


In [10]:
# [LEARNER_FILL] 难度=基础+进阶 | 提示=约30-50行 | 参考实现见 enterprise_5days/student/Day4_下午_MCP与Skills.ipynb
# ============================================================
# 练习 2 | MCP Server 加权限层（基于 user_id 控制 tool 可见性）
# ============================================================
#
# 【基础】（人人必做，10 min）
#   build_basic_server()：建含 2 tool (read_orders, get_stats) 的 EduMCPServer
#
# 【进阶】（技术学员选做，15 min）
#   build_server_with_auth()：admin 全开放；viewer 只能调 read_*
#   实现 server.set_auth_check(fn) 校验
# ============================================================

def build_basic_server():
    """【基础】2-tool server"""
    # ↓↓↓ 【基础】填空（约 8 行）↓↓↓
    server = EduMCPServer(name="exercise-server")
    def read_orders():
        """List recent orders"""
        return json.dumps([{"id": "O1", "amt": 100}, {"id": "O2", "amt": 250}])
    def get_stats():
        """Get current stats"""
        return json.dumps({"orders_today": 42, "active_users": 128})
    server.add_tool(tool_from_function(read_orders))
    server.add_tool(tool_from_function(get_stats))
    return server
    # ↑↑↑ 【基础】结束 ↑↑↑


def build_server_with_auth():
    """【进阶】admin 全开放，viewer 只能 read_*"""
    # ↓↓↓ 【进阶】填空（约 16 行）↓↓↓
    server = EduMCPServer(name="auth-server")
    def read_orders():
        """List orders"""
        return json.dumps([{"id": "O1"}])
    def write_order(item: str, qty: int):
        """Create order"""
        return json.dumps({"created": item, "qty": qty})
    def delete_user(user_id: str):
        """Delete user"""
        return json.dumps({"deleted": user_id})
    for fn in [read_orders, write_order, delete_user]:
        server.add_tool(tool_from_function(fn))

    USER_ROLES = {"alice": "admin", "bob": "viewer"}
    def auth_check(user_id, action):
        role = USER_ROLES.get(user_id, "guest")
        if role == "admin":
            return True
        if role == "viewer":
            return action.startswith("read_")
        return False
    server.set_auth_check(auth_check)
    return server
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】build_basic_server"); print("=" * 56)
    try:
        srv = build_basic_server()
        client = EduMCPClient(user_id="any")
        client.connect(srv)
        tools = client.list_all_tools()
        assert len(tools) == 2
        print(f"  Tools: {[t['name'] for t in tools]}")
        result = client.call(srv.name, "read_orders")
        print(f"  read_orders() → {result}")
        print("OK 基础通过\n")
    except NotImplementedError:
        print("SKIP 基础未实现\n"); return
    except Exception as e:
        print(f"FAIL 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】build_server_with_auth"); print("=" * 56)
    try:
        srv = build_server_with_auth()
        admin = EduMCPClient(user_id="alice"); admin.connect(srv)
        viewer = EduMCPClient(user_id="bob"); viewer.connect(srv)
        admin_tools = [t["name"] for t in admin.list_all_tools()]
        viewer_tools = [t["name"] for t in viewer.list_all_tools()]
        print(f"  admin 可见: {admin_tools}")
        print(f"  viewer 可见: {viewer_tools}")
        assert "delete_user" in admin_tools
        assert "delete_user" not in viewer_tools
        try:
            viewer.call(srv.name, "write_order", item="A", qty=1)
            print("  NO viewer 不应能调 write_order")
        except PermissionError:
            print("  viewer 调 write_order → 被拒 OK")
        print("OK 进阶通过")
    except NotImplementedError:
        print("SKIP 进阶跳过")
    except Exception as e:
        print(f"FAIL 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】build_basic_server
  Tools: ['read_orders', 'get_stats']
  read_orders() → [{"id": "O1", "amt": 100}, {"id": "O2", "amt": 250}]
OK 基础通过

【进阶】build_server_with_auth
  admin 可见: ['read_orders', 'write_order', 'delete_user']
  viewer 可见: ['read_orders']
  viewer 调 write_order → 被拒 OK
OK 进阶通过


### MCP 部分小结

- **Tools** = 函数 + JSON schema；用 `tool_from_function` 自动生成
- **Resources** = LLM 可读数据源；可静态可动态
- **Prompts**（一句话提）= 复用模板，类似 jinja2 但参数化更轻
- **Server + Auth**：生产场景用 `set_auth_check` 按用户限制 tool 可见性

下半场进入 **Skills**——如果说 MCP 解决『LLM 能调什么』，Skills 解决『LLM 知道何时怎么做什么』。


## 6. 收尾：你现在拥有什么

读到这里，你应该能理解三件事：

第一，**MCP 不是新东西，它是把"工具调用"从 N×M 集成问题降维成 N+M 的协议层**。这种模式在工程史上反复出现：通过统一接口降低适配成本，让生态可以分工演进。

第二，**MCP 的具体技术选型——stdin/stdout + JSON-RPC + 三件套——看起来朴素，但每个决定都有理由**：进程隔离做安全、JSON-RPC 跨语言、三件套对应不同权限粒度。读懂这些选型背后的 trade-off，比记住协议字段名重要得多。

第三，**权限层和 timeout/kill 是必补的两课**——不补的话第一次事故就够受。

下一节 [App6 Skills](./App6_Skills_Pack.ipynb) 看 Anthropic 的 Skill 格式：MCP 解决"工具怎么跨 LLM 通用"，Skill 解决"能力怎么跨团队复用"——两者一起构成工程化 Agent 的"工具 + 能力"双轮。